In [0]:
%sql
MERGE INTO new_catlog.intellibi_gold.RefinedCustomer AS tgt
USING (

WITH latest_cust_silver AS
(
    SELECT *
    FROM new_catlog.intellibi_silver.cleansedcustomer
    WHERE ingest_ts > (
            SELECT coalesce(MAX(start_effective_ts),to_timestamp('1990-01-01','yyyy-MM-dd'))
            FROM new_catlog.intellibi_gold.RefinedCustomer
        )
),

silver_gold_rec AS
(
SELECT  s.*,

g.customer_sk AS customer_sk,
g.is_active As is_active,
g.rec_version AS rec_version ,
g.start_effective_ts AS start_effective_ts,
g.end_effective_ts AS end_effective_ts,

CASE 
    WHEN g.customer_sk IS NULL THEN 1 
    ELSE g.rec_version + 1 
END AS new_rec_version ,

CASE 
    WHEN g.customer_sk IS NULL THEN 'NEW'

    WHEN s.email <> g.email 
    OR s.address <> g.address 
    OR s.phone <> g.phone
    OR s.city <> g.city 
    OR s.state <> g.state 
    OR s.country <> g.country 
    OR s.zip_code <> g.zip_code

    THEN 'CHANGE'

    ELSE 'NO_CHANGE' 
END AS rec_flag

FROM latest_cust_silver s

LEFT JOIN new_catlog.intellibi_gold.RefinedCustomer g
ON s.customer_id = g.customer_id
),

insert_flag AS
(
    SELECT 
    NULL AS cust_merge_key,

    customer_id,
    customer_name,
    email,
    phone,
    address,
    city,
    state,
    country,
    zip_code,
    segment,
    ingest_ts,
    customer_sk,
    is_active,
    rec_version,
    start_effective_ts,
    end_effective_ts,
    new_rec_version,

    rec_flag,
    'INSERT' AS merge_flag

    FROM silver_gold_rec 
    WHERE rec_flag IN ('NEW','CHANGE')
),

changed_flag AS
(
    SELECT 
    customer_id AS cust_merge_key,

    customer_id,
    customer_name,
    email,
    phone,
    address,
    city,
    state,
    country,
    zip_code,
    segment,
    ingest_ts,
    customer_sk,
    is_active,
    rec_version,
    start_effective_ts,
    end_effective_ts,
    new_rec_version,

    rec_flag,
    'UPDATE' AS merge_flag

    FROM silver_gold_rec 
    WHERE rec_flag IN ('CHANGE')
)

SELECT * FROM changed_flag

UNION ALL

SELECT * FROM insert_flag

) src

ON tgt.customer_id = src.cust_merge_key

WHEN MATCHED 
AND src.merge_flag = 'UPDATE' 

THEN UPDATE SET 
    is_active ='N',
    end_effective_ts = current_timestamp()

WHEN NOT MATCHED 
AND src.merge_flag = 'INSERT' 

THEN INSERT 
(
    customer_id,
    customer_name,
    email,
    phone,
    address,
    city,
    state,
    country,
    zip_code,
    segment,
    is_active,
    rec_version,
    start_effective_ts,
    end_effective_ts
)

VALUES 
(
    src.customer_id,
    src.customer_name,
    src.email,
    src.phone,
    src.address,
    src.city,
    src.state,
    src.country,
    src.zip_code,
    src.segment,
    'Y',
    src.new_rec_version,
    current_timestamp(),
    to_timestamp('9999-12-31','yyyy-MM-dd')
);